In [1]:
!apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 124 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 2s (335 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current u

In [ ]:
import subprocess
import time
import os

# 1. Matikan paksa ollama yang lama agar tidak bentrok
print("Membersihkan server lama...")
os.system("pkill ollama")
time.sleep(2)

# 2. Atur perizinan supaya terbuka untuk koneksi luar & Cloudflare
env = os.environ.copy()
env["OLLAMA_ORIGINS"] = "*"
env["OLLAMA_HOST"] = "0.0.0.0"

# 3. Jalankan ulang Ollama
print("Menghidupkan ulang server Ollama dengan akses publik...")
log_file = open("ollama.log", "w")
process = subprocess.Popen(["ollama", "serve"], env=env, stdout=log_file, stderr=log_file)
time.sleep(3)

print("✅ Selesai! Server Ollama sudah berjalan.")

Membersihkan server lama...
Menghidupkan ulang server Ollama dengan akses publik...


In [ ]:
import subprocess
import time
import re

print("🚀 Menjalankan Cloudflare Tunnel untuk Ollama (Port 11434)...")
log_file = open("cloudflared.log", "w")

# Menjalankan tunnel di background dan mengarahkannya ke localhost:11434
process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:11434"], 
    stdout=log_file, 
    stderr=log_file
)

# Tunggu sekitar 8 detik agar Cloudflare selesai men-generate URL public
print("⏳ Menunggu URL dari Cloudflare...")
time.sleep(8)

# Membaca file log untuk mengekstrak URL
with open("cloudflared.log", "r") as f:
    log_content = f.read()

# Mencari link yang berakhiran .trycloudflare.com
url_match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)

if url_match:
    print("\n✅ BERHASIL! Ollama Anda bisa diakses dari luar melalui URL berikut:")
    print(f"🔗 {url_match.group(0)}")
    print("\n(Copy URL di atas dan gunakan sebagai 'base_url' di aplikasi AI Anda)")
else:
    print("\n⚠️ URL belum muncul di log. Silakan tunggu beberapa detik lagi, lalu buka file 'cloudflared.log' secara manual untuk mengecek URL-nya.")

In [ ]:
!wget -L "https://huggingface.co/0bserverx/Qwen3.8-27B-Heretic-Abliterated-Uncensored-GGUF/resolve/main/RVN-Q4_K_M.gguf?download=true" -O model.gguf
!echo "FROM ./model.gguf" > Modelfile
!ollama create llmy -f Modelfile

Testing Ollama

In [ ]:
!ollama run llmy "Halo, tes apakah kamu berfungsi?"

In [ ]:
import requests

url = url_match.group(0)+"/api/generate"
data = {
    "model": "llmy",
    "prompt": "Halo, tes apakah kamu berfungsi? Ceritakan 1 fakta unik tentang AI.",
    "stream": False
}

response = requests.post(url, json=data)
print(response.json().get('response', 'Tidak ada respons'))

In [ ]:
import requests

url = url_match.group(0)+"/v1/chat/completions" # Endpoint kompatibel OpenAI
data = {
    "model": "llmy",
    "messages": [
        {"role": "system", "content": "Kamu adalah asisten AI yang cerdas dan asyik."},
        {"role": "user", "content": "Kenapa CPU mu naik? ga gpu aja"}
    ]
}

response = requests.post(url, json=data)
# Cara mengekstrak jawabannya juga mengikuti gaya JSON OpenAI
print(response.json()['choices'][0]['message']['content'])